# FIAP Global Solution 2026.2, CitizenTech
## Golpe Zero - Especialização com LoRA (PEFT) vs RAG simples
**Disciplina:** Generative AI for Engineering · **Professor:** Francisco Elânio Bezerra

Este caderno treina um adapter LoRA de verdade sobre o `Qwen2.5-0.5B-Instruct` e compara, nos 35 casos held-out, a especialização paramétrica (LoRA) com a especialização contextual (RAG simples) na mesma base de 0.5B.

1. Fundamentação do LoRA · 2. Dataset de treino · 3. Hiperparâmetros · 4. Treino · 5. Adapter exportado · 6. Placar LoRA vs RAG simples · 7. Conclusão

### 1. Fundamentação do LoRA
Em vez de atualizar a matriz de pesos $W \in \mathbb{R}^{d \times k}$, o LoRA congela $W$ e aprende uma atualização de posto baixo:

$$W' = W + \frac{\alpha}{r} B A, \quad B \in \mathbb{R}^{d \times r},\; A \in \mathbb{R}^{r \times k},\; r \ll \min(d, k)$$

Com $r=8$ e $\alpha=16$ aplicados a `q_proj` e `v_proj`, treina-se uma fração mínima dos parâmetros.

### 2. Dataset de treino (sintético, disjunto do gabarito)

In [ ]:
import json
from notebooks.dataset_lora import gerar_dataset, verificar_vazamento

dados = gerar_dataset(7)
from collections import Counter
print('exemplos:', len(dados))
print('mensagens de treino idênticas ao gabarito:', verificar_vazamento(dados))
print('distribuição de vereditos:', Counter(json.loads(d['output'])['veredito'] for d in dados))
print('\nExemplo de treino:')
print(dados[0]['input'])
print(dados[0]['output'])

### 3. Hiperparâmetros LoRA

In [ ]:
from peft import LoraConfig, TaskType

cfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
                 task_type=TaskType.CAUSAL_LM, target_modules=['q_proj', 'v_proj'])
print(cfg)

### 4. Treino real (perda só nos tokens da resposta JSON)

In [ ]:
from notebooks.treinar_lora import executar_fine_tuning

resumo = executar_fine_tuning()
print(json.dumps(resumo, indent=2, ensure_ascii=False))

### 5. Adapter exportado

In [ ]:
from pathlib import Path

for arq in sorted(Path('adapters/golpe_zero_lora').glob('*')):
    print(f'{arq.name:<30} {arq.stat().st_size/1024:>9.1f} KB')
cfg_salvo = json.load(open('adapters/golpe_zero_lora/adapter_config.json'))
print('base_model_name_or_path:', cfg_salvo['base_model_name_or_path'])

### 6. Placar held-out: base zero-shot · base + RAG simples · base + LoRA
Os 35 casos (25 principais + 10 de estresse) não foram vistos no treino.

In [ ]:
from avaliacao.benchmark_lora import executar

placar = executar()